# ray-parametric-form — ex2: evaluate a batch of rays at one parameter

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `ray-parametric-form`. Running the final beacon cell reports progress against the `Geometry: Ray parametric form` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Geometry: Ray parametric form` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`ray-parametric-form`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "ray-parametric-form"
DD_SUBTOPIC = "Geometry: Ray parametric form"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Ray parametric form — quick refresher

A ray in 3-D is specified by an **origin** `O` (a point) and a **direction** `D` (a vector). Every point on the ray is

```
R(u) = O + u * D,  u >= 0
```

**Anatomy:**
- `u = 0` returns the origin.
- `u > 0` walks forward along `D`.
- `u < 0` would walk *backward* — by convention, real rays restrict `u >= 0`.

**Storage convention used in ARENA.** A ray is a `(2, 3)` tensor: row 0 is `O`, row 1 is `D`. A batch of `B` rays is a `(B, 2, 3)` tensor. To evaluate `B` rays at a single scalar `u`, you compute `origin + u * direction` with ordinary broadcasting; to evaluate one ray at `M` parameter values, you broadcast `(M,)` against `(3,)`.

**Why `D` is not required to be unit-length.** If `||D|| != 1` then `u` is not a metric distance — it's a parameter along `D`. ARENA leaves `D` un-normalized everywhere, so `u` is dimensionless.

### Exercise 2 — evaluate a batch of rays at one parameter

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the parametric ray equation `R(u) = O + u * D` across a batch of rays at a single scalar parameter `u`, returning one endpoint per ray.
> Keywords: ray, batched, parametric, visualization
> ```

**KCs targeted:** `ray-eval-broadcast-batch`, `ray-origin-direction-storage`

Implement `ex2_eval_ray_batch(rays, u)`.

- `rays` has shape `(B, 2, 3)` — row 0 of each `(2, 3)` block is `O_b`, row 1 is `D_b`.
- `u` is a Python `float`.
- Return shape `(B, 3)`: the point `O_b + u * D_b` for each ray.

**Hint.** Slice off origins and directions with `rays[:, 0]` and `rays[:, 1]` (each `(B, 3)`). Then the arithmetic is plain broadcasting — no `unsqueeze` needed because `u` is scalar.

The visualization plots all `B` endpoints in the X-Z plane next to their origins, drawing the connecting segments so you can see the fan of rays.

In [ ]:
def ex2_eval_ray_batch(rays: Tensor, u: float) -> Tensor:
    O = rays[:, 0]   # (B, 3)
    D = rays[:, 1]   # (B, 3)
    return O + u * D


<details><summary>Solution</summary>

```python
def ex2_eval_ray_batch(rays: Tensor, u: float) -> Tensor:
    O = rays[:, 0]   # (B, 3)
    D = rays[:, 1]   # (B, 3)
    return O + u * D
```

**Why no `unsqueeze` here.** `u` is a Python scalar — multiplying a tensor by it is rank-preserving (`(B, 3) * scalar → (B, 3)`). Compare with ex1, where the parameter sweep was a `(M,)` tensor and we had to add an axis to align.

**Slicing vs unbind.** `rays[:, 0]` and `rays[:, 1]` create views (no copy). `O, D = rays.unbind(dim=1)` is equivalent and arguably cleaner — it makes the two-row decomposition explicit. Both compile to the same arithmetic.

**Generalizing.** To allow per-ray parameters (an `(B,)` tensor of `u` values instead of one scalar), you'd write `O + us.unsqueeze(-1) * D` — the same `(B, 1) * (B, 3)` trick from ex1, just with the batch axis playing the role of `M`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()